# Local CIFAR-10 method smoke test

This notebook checks that each registered method can run from a fresh copy of the same pretrained ResNet-18 and returns an evaluable model. To keep it quick, training uses fixed subsets (2,048 retain, 256 forget, 512 validation). Test accuracy uses the full test split; the loss-MIA diagnostic uses an equal-size 256-example test sample.

The table is a **single-run diagnostic**, not the official competition score. Use `04_official_score.ipynb` for multi-seed F and utility ratios.


In [4]:
from pathlib import Path
import copy
import gc
import sys
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

repo_root = Path.cwd()
if not (repo_root / "src" / "unlearn" / "data.py").is_file():
    repo_root = repo_root.parent
if not (repo_root / "src" / "unlearn" / "data.py").is_file():
    raise FileNotFoundError(f"Could not find src/unlearn/data.py from {Path.cwd()}")
sys.path.insert(0, str(repo_root / "src"))

from unlearn.data import get_cifar10_loaders
from unlearn.device import DEVICE
from unlearn.metrics import accuracy_eval, mia_accuracy
from unlearn.models import load_cifar10_model
from unlearn.methods import METHODS

print("Running on device:", str(DEVICE).upper())
print("Registered methods:", ", ".join(METHODS))


Running on device: MPS
Registered methods: Amnesiacs_6, Fauchan_1, Kookmin_2, Seif_3, finetune


In [5]:
loaders = get_cifar10_loaders(
    root=str(repo_root / "Notebooks" / "data"),
    forget_idx_path=str(repo_root / "Notebooks" / "forget_idx.npy"),
    batch_size=128,
    num_workers=0,
    seed=42,
)

def sample_loader(loader, count, seed, *, shuffle=False):
    dataset = loader.dataset
    count = min(count, len(dataset))
    indices = np.random.default_rng(seed).choice(len(dataset), count, replace=False)
    dataset = Subset(dataset, np.sort(indices).tolist())
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(
        dataset,
        batch_size=loader.batch_size or 128,
        shuffle=shuffle,
        num_workers=0,
        generator=generator,
    )

SMOKE_SEED = 2026
retain_smoke = sample_loader(loaders["retain"], 2048, SMOKE_SEED, shuffle=True)
forget_smoke = sample_loader(loaders["forget"], 256, SMOKE_SEED + 1)
val_smoke = sample_loader(loaders["val"], 512, SMOKE_SEED + 2)
test_mia = sample_loader(loaders["test"], len(forget_smoke.dataset), SMOKE_SEED + 3)
test_full = loaders["test"]
original_model = load_cifar10_model(repo_root / "weights_resnet18_cifar10.pth")

print(f"Smoke subset sizes: retain={len(retain_smoke.dataset)}, "
      f"forget={len(forget_smoke.dataset)}, validation={len(val_smoke.dataset)}, "
      f"MIA test={len(test_mia.dataset)}")
print(f"Full test size for accuracy: {len(test_full.dataset)}")


Smoke subset sizes: retain=2048, forget=256, validation=512, MIA test=256
Full test size for accuracy: 5000


In [6]:
# Keep these settings explicit so each method's local approximation is visible.
method_kwargs = {
    "Amnesiacs_6": {"temperature": 5.0, "warmup_epochs": 3, "retain_epochs": 3},
    "Fauchan_1": {"epochs": 8, "retain_bs": 256, "temperature": 1.15, "device": DEVICE},
    "Seif_3": {"device": DEVICE},
}

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    elif DEVICE.type == "mps":
        torch.mps.synchronize()

results = []
models = {}
for name, method_fn in METHODS.items():
    torch.manual_seed(SMOKE_SEED)
    model_copy = copy.deepcopy(original_model).to(DEVICE)
    synchronize()
    started = time.perf_counter()
    result_model = method_fn(
        model_copy,
        retain_smoke,
        forget_smoke,
        val_smoke,
        **method_kwargs.get(name, {}),
    )
    synchronize()
    elapsed = time.perf_counter() - started
    if not isinstance(result_model, torch.nn.Module):
        raise TypeError(f"{name} returned {type(result_model).__name__}, not a model")

    row = {
        "method": name,
        "retain_acc_subset": accuracy_eval(result_model, retain_smoke),
        "test_acc_full": accuracy_eval(result_model, test_full),
        "forget_acc_subset": accuracy_eval(result_model, forget_smoke),
        "runtime_sec": elapsed,
        "loss_mia_acc": mia_accuracy(result_model, forget_smoke, test_mia, seed=SMOKE_SEED),
    }
    results.append(row)
    models[name] = result_model
    del model_copy, result_model
    gc.collect()

print(f"{'method':<16} {'retain(sub)':>12} {'test(full)':>12} "
      f"{'forget(sub)':>12} {'runtime(s)':>12} {'loss-MIA':>10}")
for row in results:
    print(f"{row['method']:<16} {row['retain_acc_subset'] * 100:>11.2f}% "
          f"{row['test_acc_full'] * 100:>11.2f}% "
          f"{row['forget_acc_subset'] * 100:>11.2f}% "
          f"{row['runtime_sec']:>12.2f} {row['loss_mia_acc']:>10.3f}")


-----------------------------------
-----------------------------------
Epoch [1/4], Loss: 0.8582
Epoch [2/4], Loss: 0.7008
Epoch [3/4], Loss: 0.5413
Epoch [4/4], Loss: 0.5089
method            retain(sub)   test(full)  forget(sub)   runtime(s)   loss-MIA
Amnesiacs_6            23.78%       20.08%       21.48%         3.79      0.510
Fauchan_1             100.00%       86.40%       76.95%         5.61      0.415
Kookmin_2              93.60%       63.62%       64.45%         3.92      0.506
Seif_3                 33.98%       28.22%       26.56%         4.14      0.429
finetune               87.55%       59.90%       59.38%         3.39      0.471


## Reading the smoke-test table

- **Retain / forget accuracy** are measured on the fixed small smoke subsets; they can vary with the subset seed.
- **Test accuracy** uses the entire held-out test split.
- **Runtime** times only the method call (with accelerator synchronization), not accuracy or MIA evaluation.
- **Loss-MIA** tries to distinguish forget examples from an equal number of test examples using their losses. It is only a diagnostic; it is not official forgetting quality F.
- Every method receives a fresh copy of the same starting model. This notebook is meant to find interface/device/data errors quickly. For final method comparisons, use identical N, seeds, and forget examples in `04_official_score.ipynb`.

The Amnesiacs validation filter defaults to labels 0 and 1 to mirror the competition age-group setup. On CIFAR-10 those are object labels, so this is a code-path smoke test, not a faithful CASIA-SURF experiment.
